# MCP 与 LangGraph：用模型上下文协议扩展智能体

**挑战：** 你希望让 LangGraph 智能体访问外部工具与资源——例如网页搜索、数据库和 API——但如果直接在智能体代码里管理这些集成，就会形成紧耦合并带来维护负担。每增加一种能力都需要改代码，而且不同工具往往还有不同接口。

**解决方案：** **模型上下文协议（Model Context Protocol，MCP）** 为 AI 应用暴露工具和资源提供了一套标准方式。借助 `langchain-mcp-adapters`，你可以把 MCP Server 无缝接入 LangGraph 智能体，让智能体获得强大的外部能力，同时不必把大量集成代码塞进智能体本身。

本 Notebook 将演示：
1. **MCP 原理（MCP Principles）：** MCP 是什么，以及为什么重要
2. **简单 Server：** 构建一个基础的文件管理 MCP Server
3. **LangGraph 集成：** 在 StateGraph 智能体中使用 MCP 工具
4. **生产级示例：** 构建带有类型化输入输出、安全约束与治理机制的 SQLite 库存管理器
5. **第三方集成：** 快速接入外部 MCP Server

<br>

## 最佳实践

### 1. **传输方式选择（Transport Selection）**
- **stdio**：更适合本地开发和单用户场景
- **HTTP / Streamable HTTP**：更适合 Web Server 和多用户场景

### 2. **并发安全（Concurrency Safety）**
- 并发调用工具时使用“每次调用独立连接”，不要共享全局连接
- 为 SQLite 启用 WAL 模式，以改善并发能力
- 使用原子更新（atomic update）避免竞态条件（race condition）

### 3. **治理（Governance）**
- 在 MCP Server 边界执行策略，而不是把策略约束交给模型自己遵守
- 在多智能体系统中，MCP Server 可以承担治理层（governance layer）的角色

### 4. **类型化输入输出（Typed IO）**
- 输入和输出都使用 Pydantic 模型，以形成显式契约
- 这样可以减少静默失败，并让工具行为更可预测

## 安装

首先安装所需依赖包：

In [ ]:
# Install required packages
%pip install langchain-mcp-adapters langgraph langchain-core langchain-openai mcp fastmcp pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.2/416.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.8/199.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331

# 理解 MCP

**模型上下文协议（Model Context Protocol，MCP）** 是一种开放协议，用于标准化 AI 应用与外部工具和资源之间的交互方式。可以把它理解为一个通用适配层：智能体只需要理解统一接口，就能调用外部能力，而不必知道背后的具体实现。

### 核心概念

- **MCP Server：** 提供工具与资源，例如文件操作、数据库访问和网页搜索
- **MCP Client：** 连接 Server，并把其能力暴露给 AI 应用
- **传输层（Transports）：** 通信方式，例如 stdio、HTTP、SSE、Streamable HTTP
- **工具（Tools）：** 智能体可以调用的函数，例如 `read_file`、`search_web`

### 为什么 MCP 重要

- **解耦（Decoupling）：** 工具运行在独立 Server 中，而不是直接嵌在智能体代码里
- **复用（Reusability）：** 一个 MCP Server 可以同时服务多个智能体
- **标准化（Standardization）：** 不同工具通过一致接口暴露能力
- **治理（Governance）：** MCP Server 可以在模型之外执行策略和系统不变量（invariants）

# 简单 MCP Server：文件管理器

先从一个简单的文件管理 MCP Server 开始理解基本模式：用 Pydantic 模型定义工具输入输出，再通过 FastMCP 对外暴露工具。这个示例展示：

- 使用 Pydantic 模型实现类型化输入输出（Typed IO）
- 使用 FastMCP 定义基础工具
- 标准的 MCP Server 组织方式

In [ ]:
# 创建一个简单的文件管理 MCP Server
file_manager_code = '''
"""
Simple File Manager MCP Server

A minimal example showing:
- Typed IO with Pydantic models
- Basic file operations
- FastMCP server setup
"""

from pathlib import Path
from typing import Optional
from pydantic import BaseModel, Field
from mcp.server.fastmcp import FastMCP

# 用 Pydantic 模型定义类型化输入输出（Typed IO）
class ReadFileRequest(BaseModel):
    """Read a file."""
    file_path: str = Field(..., description="Path to file to read")

class WriteFileRequest(BaseModel):
    """Write content to a file."""
    file_path: str = Field(..., description="Path to file to write")
    content: str = Field(..., description="Content to write")

class ListFilesRequest(BaseModel):
    """List files in a directory."""
    directory: str = Field(..., description="Directory path")
    pattern: Optional[str] = Field(None, description="Optional glob pattern (e.g., '*.py')")

# 创建 MCP Server
mcp = FastMCP("FileManager")

@mcp.tool()
def read_file(request: ReadFileRequest) -> str:
    """Read the contents of a file."""
    path = Path(request.file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {request.file_path}")
    return path.read_text(encoding="utf-8")

@mcp.tool()
def write_file(request: WriteFileRequest) -> dict:
    """Write content to a file. Creates file if it doesn't exist."""
    path = Path(request.file_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(request.content, encoding="utf-8")
    return {"status": "success", "file_path": str(path), "bytes_written": len(request.content)}

@mcp.tool()
def list_files(request: ListFilesRequest) -> list[str]:
    """List files in a directory."""
    dir_path = Path(request.directory)
    if not dir_path.exists():
        raise FileNotFoundError(f"Directory not found: {request.directory}")

    if request.pattern:
        files = list(dir_path.glob(request.pattern))
    else:
        files = list(dir_path.iterdir())

    return [str(f.relative_to(dir_path)) for f in files if f.is_file()]

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# 写出 Server 文件
from pathlib import Path
file_manager_path = Path("file_manager_server.py")
file_manager_path.write_text(file_manager_code)
print(f"Created {file_manager_path}")


Created file_manager_server.py


# 连接 MCP Server，并在 LangGraph 中使用

接下来连接刚才创建的 MCP Server，并把它集成进 LangGraph。MCP 的价值在这里会更直观：智能体能够使用外部工具，而不需要了解这些工具的具体实现方式。

## 配置 API Key（也可以使用环境变量）

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
EXA_API_KEY = os.environ.get("EXA_API_KEY", "")

## 为 `OutStream.fileno` 打补丁

这个 Monkey Patch 会保留 Jupyter 原有的打印行为，只让 `fileno()` 在 Jupyter Notebook 中不再抛出错误。**注意：** 如果是在普通 Python 脚本中运行，则不需要这段补丁。

In [ ]:
from ipykernel.iostream import OutStream

def _safe_fileno(self):
    name = getattr(self, "name", "")
    if "stderr" in name:
        return 2
    return 1

OutStream.fileno = _safe_fileno


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.chat_models import init_chat_model

# 初始化模型
try:
    model = init_chat_model("openrouter:anthropic/claude-3.5-haiku")
except:
    model = init_chat_model("openai:gpt-4o-mini")

# 创建 MCP Client，并连接文件管理 Server
client = MultiServerMCPClient(
    {
        "file_manager": {
            "command": "python",
            "args": [os.path.abspath("file_manager_server.py")],
            "transport": "stdio",
        }
    }
)


# 从 MCP Server 加载工具
tools = await client.get_tools()

print(f" Loaded {len(tools)} tools from MCP server:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

# 定义智能体节点
def call_model(state: MessagesState):
    """Call the model with tools bound."""
    response = model.bind_tools(tools).invoke(state["messages"])
    return {"messages": [response]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "call_model")
builder.add_conditional_edges("call_model", tools_condition)
builder.add_edge("tools", "call_model")

graph = builder.compile()
print("\n LangGraph agent created with MCP tools")

 Loaded 3 tools from MCP server:
  - read_file: Read the contents of a file.
  - write_file: Write content to a file. Creates file if it doesn't exist.
  - list_files: List files in a directory.

 LangGraph agent created with MCP tools


## 用文件操作测试智能体

In [ ]:

response = await graph.ainvoke({
    "messages": [("user", "Create a file called 'test.txt' with content 'Hello from MCP!', then read it back.")]
})

# 展示响应
from langchain_core.messages import AIMessage
for message in response["messages"]:
    if isinstance(message, AIMessage):
        print("Agent Response:")
        print(message.content)
        if message.tool_calls:
            print("\nTool Calls:")
            for tool_call in message.tool_calls:
                print(f"  - {tool_call['name']}({tool_call['args']})")

Agent Response:


Tool Calls:
  - write_file({'request': {'file_path': 'test.txt', 'content': 'Hello from MCP!'}})
  - read_file({'request': {'file_path': 'test.txt'}})
Agent Response:
The file 'test.txt' has been created with the content 'Hello from MCP!'. When read back, the content is: **Hello from MCP!**


# 生产级示例：库存管理器

下面看一个更接近生产环境的例子：基于 SQLite 的库存管理器。它展示了 MCP Server 如何提供持久化状态、安全保证和策略执行。

这个库存管理器体现了企业级常见模式：
- **业务部门拥有系统** → 通过 MCP Server 暴露能力
- **智能体系统消费能力** → 使用稳定、类型明确的工具契约
- **MCP = 互操作层（Interop Layer）** → 多个多智能体系统（MAS）可以在不共享内部代码的情况下接入同一能力
- **边界治理（Governance at Boundary）** → 策略在 Server 端执行，而不是依赖模型自行遵守


In [ ]:
# 创建更接近生产环境的库存管理 MCP Server
inventory_server_code = '''
"""
SQLite Inventory Manager MCP Server

Production patterns:
- Typed IO with Pydantic models (input and output)
- Safety: Atomic updates prevent negative stock
- Concurrency: Per-call connections, SQLite WAL mode
- Governance: Policy gates enforced at MCP boundary
"""

import sqlite3
import json
from pathlib import Path
from typing import Optional, List
from contextlib import contextmanager
from pydantic import BaseModel, Field, field_validator
from mcp.server.fastmcp import FastMCP

# ============================================================================
# Pydantic 模型
# ============================================================================

class ItemCreate(BaseModel):
    """Create a new inventory item."""
    name: str = Field(..., min_length=1, max_length=200)
    initial_stock: int = Field(..., ge=0)
    unit_price: float = Field(..., ge=0)
    category: Optional[str] = None

class StockAdjustment(BaseModel):
    """Adjust stock quantity."""
    item_id: int = Field(..., gt=0)
    quantity: int = Field(..., description="Positive adds, negative removes")
    reason: str = Field(..., min_length=1)
    approval_token: Optional[str] = None

    @field_validator("quantity")
    @classmethod
    def quantity_not_zero(cls, v):
        if v == 0:
            raise ValueError("Quantity must be non-zero")
        return v

class ItemQuery(BaseModel):
    """Query items."""
    item_id: Optional[int] = None
    category: Optional[str] = None
    low_stock_threshold: Optional[int] = None

class ItemOut(BaseModel):
    """Item output model."""
    id: int
    name: str
    stock: int
    unit_price: float
    category: Optional[str]

# ============================================================================
# 数据库初始化
# ============================================================================

DB_PATH = Path("inventory.db")

def init_database():
    """Initialize database schema."""
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("PRAGMA foreign_keys=ON")

    conn.execute("""
        CREATE TABLE IF NOT EXISTS items (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
            unit_price REAL NOT NULL CHECK(unit_price >= 0),
            category TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS audit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            item_id INTEGER,
            event_type TEXT NOT NULL,
            event_data TEXT NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (item_id) REFERENCES items(id)
        )
    """)

    conn.commit()
    conn.close()

init_database()

def connect():
    """Create a new connection for each tool call (safe for concurrency)."""
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("PRAGMA foreign_keys=ON")
    return conn

@contextmanager
def transaction():
    """Context manager for transactions with per-call connections."""
    conn = connect()
    try:
        conn.execute("BEGIN")
        yield conn
        conn.execute("COMMIT")
    except Exception:
        conn.execute("ROLLBACK")
        raise
    finally:
        conn.close()

def log_event(conn, item_id: Optional[int], event_type: str, event_data: dict):
    """Append event to audit log using active connection."""
    conn.execute(
        "INSERT INTO audit_log (item_id, event_type, event_data) VALUES (?, ?, ?)",
        (item_id, event_type, json.dumps(event_data))
    )

# ============================================================================
# 策略执行
# ============================================================================

def enforce_policy(adjustment: StockAdjustment):
    """Enforce business policies - MCP servers are the governance boundary."""
    # 策略：大幅减库存需要审批
    if adjustment.quantity < -100:
        if not adjustment.approval_token or adjustment.approval_token != "APPROVED_BY_MANAGER":
            raise ValueError(
                f"Large stock reduction ({abs(adjustment.quantity)} units) requires approval_token."
            )

    # 策略：负向库存调整必须引用订单
    if adjustment.quantity < 0:
        if not adjustment.reason.startswith("ORDER:"):
            raise ValueError(
                "Negative adjustments must include order reference. Format: 'ORDER: <id> - <desc>'"
            )

# ============================================================================
# MCP Server
# ============================================================================

mcp = FastMCP("InventoryManager")

@mcp.tool()
def create_item(item: ItemCreate) -> ItemOut:
    """Create a new inventory item."""
    with transaction() as conn:
        cursor = conn.execute(
            "INSERT INTO items (name, stock, unit_price, category) VALUES (?, ?, ?, ?)",
            (item.name, item.initial_stock, item.unit_price, item.category)
        )
        item_id = cursor.lastrowid

        log_event(conn, item_id, "item_created", {
            "name": item.name,
            "initial_stock": item.initial_stock
        })

        row = conn.execute(
            "SELECT id, name, stock, unit_price, category FROM items WHERE id = ?",
            (item_id,)
        ).fetchone()

        return ItemOut(
            id=row["id"],
            name=row["name"],
            stock=row["stock"],
            unit_price=row["unit_price"],
            category=row["category"]
        )

@mcp.tool()
def adjust_stock(adjustment: StockAdjustment) -> ItemOut:
    """Adjust stock quantity. Enforces no negative stock and policy gates."""
    # 执行策略门控（治理边界）
    enforce_policy(adjustment)

    with transaction() as conn:
        # 原子更新：只有在库存不会变成负数时才成功
        cur = conn.execute(
            "UPDATE items SET stock = stock + ? WHERE id = ? AND stock + ? >= 0",
            (adjustment.quantity, adjustment.item_id, adjustment.quantity)
        )

        if cur.rowcount == 0:
            exists = conn.execute("SELECT 1 FROM items WHERE id = ?", (adjustment.item_id,)).fetchone()
            if not exists:
                raise ValueError(f"Item {adjustment.item_id} not found")

            current = conn.execute("SELECT stock FROM items WHERE id = ?", (adjustment.item_id,)).fetchone()
            current_stock = current["stock"] if current else 0
            raise ValueError(
                f"Insufficient stock. Current: {current_stock}, requested: {adjustment.quantity}"
            )

        updated = conn.execute("SELECT stock FROM items WHERE id = ?", (adjustment.item_id,)).fetchone()
        new_stock = updated["stock"]

        log_event(conn, adjustment.item_id, "stock_adjusted", {
            "adjustment": adjustment.quantity,
            "new_stock": new_stock,
            "reason": adjustment.reason
        })

        row = conn.execute(
            "SELECT id, name, stock, unit_price, category FROM items WHERE id = ?",
            (adjustment.item_id,)
        ).fetchone()

        return ItemOut(
            id=row["id"],
            name=row["name"],
            stock=row["stock"],
            unit_price=row["unit_price"],
            category=row["category"]
        )

@mcp.tool()
def list_items(query: ItemQuery) -> List[ItemOut]:
    """Query inventory items."""
    conn = connect()
    try:
        conditions = []
        params = []

        if query.item_id:
            conditions.append("id = ?")
            params.append(query.item_id)
        if query.category:
            conditions.append("category = ?")
            params.append(query.category)
        if query.low_stock_threshold is not None:
            conditions.append("stock <= ?")
            params.append(query.low_stock_threshold)

        where = "WHERE " + " AND ".join(conditions) if conditions else ""

        rows = conn.execute(
            f"SELECT id, name, stock, unit_price, category FROM items {where} ORDER BY name",
            params
        ).fetchall()

        return [
            ItemOut(
                id=row["id"],
                name=row["name"],
                stock=row["stock"],
                unit_price=row["unit_price"],
                category=row["category"]
            )
            for row in rows
        ]
    finally:
        conn.close()

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# 写出 Server 文件
inventory_path = Path("inventory_server.py")
inventory_path.write_text(inventory_server_code)
print(f"Created {inventory_path}")
print("\nKey patterns demonstrated:")
print("  • Typed IO: Pydantic models for contracts")
print("  • Safety: Atomic updates prevent race conditions")
print("  • Concurrency: Per-call connections, WAL mode")
print("  • Governance: Policy gates enforced at server boundary")

Created inventory_server.py

Key patterns demonstrated:
  • Typed IO: Pydantic models for contracts
  • Safety: Atomic updates prevent race conditions
  • Concurrency: Per-call connections, WAL mode
  • Governance: Policy gates enforced at server boundary


In [ ]:
# 创建接入库存管理器的智能体
inventory_client = MultiServerMCPClient(
    {
        "inventory": {
            "command": "python",
            "args": [os.path.abspath("inventory_server.py")],
            "transport": "stdio",
        }
    }
)

inventory_tools = await inventory_client.get_tools()

def call_model_inventory(state: MessagesState):
    response = model.bind_tools(inventory_tools).invoke(state["messages"])
    return {"messages": [response]}

builder_inv = StateGraph(MessagesState)
builder_inv.add_node("call_model", call_model_inventory)
builder_inv.add_node("tools", ToolNode(inventory_tools))
builder_inv.add_edge(START, "call_model")
builder_inv.add_conditional_edges("call_model", tools_condition)
builder_inv.add_edge("tools", "call_model")

inventory_graph = builder_inv.compile()

# 演示：创建商品、调整库存，并测试安全约束与策略门控
print("=== Inventory Manager Demo ===\n")

# 1. 创建商品
print("1. Creating item...")
response1 = await inventory_graph.ainvoke({
    "messages": [("user", "Create a new item: 'Laptop Pro' with initial stock 10, price $1299.99, category 'Electronics'")]
})
print("   ✅ Item created\n")

# 2. 增加库存
print("2. Adding stock...")
response2 = await inventory_graph.ainvoke({
    "messages": [("user", "Add 5 units to item ID 1, reason: 'Received shipment'")]
})
print("   ✅ Stock increased\n")

# 3. 测试安全约束：尝试扣减超过现有库存的数量
print("3. Testing safety invariant (prevent negative stock)...")
try:
    await inventory_graph.ainvoke({
        "messages": [("user", "Remove 1000 units from item ID 1, reason: 'ORDER: TEST-001 - Test order'")]
    })
    print("   ⚠️  Safety check failed!")
except Exception as e:
    print(f"   ✅ Safety check passed: {str(e)[:60]}...\n")

# 4. 测试策略：未经审批的大幅减库存
print("4. Testing policy gate (large reduction requires approval)...")
try:
    await inventory_graph.ainvoke({
        "messages": [("user", "Reduce stock by 150 units for item ID 1, reason: 'ORDER: BULK-001 - Bulk order'")]
    })
    print("   ⚠️  Policy check failed!")
except Exception as e:
    print(f"   ✅ Policy gate enforced: {str(e)[:60]}...\n")

print("=== Key Insights ===")
print("  • MCP servers enforce safety invariants (no negative stock)")
print("  • Policy gates are enforced at the server boundary, not in the model")
print("  • Multiple agents can safely share the same database through MCP")

=== Inventory Manager Demo ===

1. Creating item...
   ✅ Item created

2. Adding stock...
   ✅ Stock increased

3. Testing safety invariant (prevent negative stock)...
   ✅ Safety check passed: Error executing tool adjust_stock: Large stock reduction (10...

4. Testing policy gate (large reduction requires approval)...
   ✅ Policy gate enforced: Error executing tool adjust_stock: Large stock reduction (15...

=== Key Insights ===
  • MCP servers enforce safety invariants (no negative stock)
  • Policy gates are enforced at the server boundary, not in the model
  • Multiple agents can safely share the same database through MCP


# 第三方 MCP Server

很多服务已经提供现成的 MCP Server，便于直接集成。下面用 Exa 的 MCP Server 做一个简短示例，为智能体增加网页搜索能力：

In [ ]:
# 示例：把库存管理器与 Exa 搜索组合起来
EXA_MCP_URL = "https://mcp.exa.ai/mcp"


if EXA_API_KEY:
    # 组合多个 MCP Server
    combined_client = MultiServerMCPClient(
        {
            "inventory": {
                "command": "python",
                "args": [os.path.abspath("inventory_server.py")],
                "transport": "stdio",
            },
            "exa": {
                "transport": "http",
                "url": EXA_MCP_URL,
                "headers": {"Authorization": f"Bearer {EXA_API_KEY}"},
            }
        }
    )

    combined_tools = await combined_client.get_tools()
    print(f"✅ Loaded {len(combined_tools)} tools from multiple MCP servers")
    print("   You can now use both inventory management and web search in the same agent")
else:
    print("⚠️  Set EXA_API_KEY to test third-party MCP integration")
    print("   Example: os.environ['EXA_API_KEY'] = 'your-key-here'")

✅ Loaded 5 tools from multiple MCP servers
   You can now use both inventory management and web search in the same agent
